# Corpus information

- Description of the chosen corpus: The "Emotion" dataset consists of English Twitter messages labeled with six basic emotions: sadness (0), joy (1), love (2), anger (3), fear (4), and surprise (5).
- Paper(s) and other published materials related to the corpus: "CARER: Contextualized Affect Representations for Emotion Recognition" (Saravia et al., 2018).
- State-of-the-art performance (best published results) on this corpus: Currently, transformer-based pretrained language models (like RoBERTa and DeBERTa) achieve the SOTA performance on this corpus, reaching an Accuracy and F1-score of around 93% - 95%.


---

## 1. Setup

In [ ]:
!pip install datasets scikit-learn pandas matplotlib

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import time
from sklearn.metrics import f1_score

---

## 2. Data download and preprocessing

### 2.1. Download the corpus

In [ ]:
dataset = load_dataset("dair-ai/emotion")

# Split into corresponding subsets
train_data = dataset['train']
val_data = dataset['validation']
test_data = dataset['test']

print(f"Train size: {len(train_data)}, Validation size: {len(val_data)}, Test size: {len(test_data)}")

Train size: 16000, Validation size: 2000, Test size: 2000


### 2.2. Preprocessing

In [ ]:
# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit on training data and transform all sets
X_train = vectorizer.fit_transform(train_data['text'])
X_val = vectorizer.transform(val_data['text'])
X_test = vectorizer.transform(test_data['text'])

# Extract labels
y_train = train_data['label']
y_val = val_data['label']
y_test = test_data['label']

print(f"Shape of X_train: {X_train.shape}")

Shape of X_train: (16000, 5000)


---

## 3. Machine learning model

### 3.1. Model training

In [ ]:
# Train a basic MLP model
mlp_basic = MLPClassifier(hidden_layer_sizes=(100,), max_iter=200, random_state=42, verbose=False)
mlp_basic.fit(X_train, y_train)

# Evaluate on Validation set
y_pred_val = mlp_basic.predict(X_val)
print("Basic MLP Validation Results:")
print(classification_report(y_val, y_pred_val, target_names=dataset['train'].features['label'].names))

Basic MLP Validation Results:
              precision    recall  f1-score   support

     sadness       0.88      0.89      0.89       550
         joy       0.87      0.90      0.88       704
        love       0.80      0.76      0.78       178
       anger       0.89      0.84      0.86       275
        fear       0.80      0.80      0.80       212
    surprise       0.80      0.78      0.79        81

    accuracy                           0.86      2000
   macro avg       0.84      0.83      0.83      2000
weighted avg       0.86      0.86      0.86      2000



### 3.2 Hyperparameter optimization

In [ ]:
configs = [
    {"name": "Small Network", "params": {'hidden_layer_sizes': (50,), 'max_iter': 200, 'random_state': 42}},
    {"name": "Deep Network", "params": {'hidden_layer_sizes': (150, 50), 'max_iter': 200, 'random_state': 42}},
    {"name": "High Learning Rate", "params": {'hidden_layer_sizes': (100,), 'learning_rate_init': 0.01, 'max_iter': 200, 'random_state': 42}}
]

best_model = None
best_f1 = 0

print("Hyperparameter Optimization Results:")
for config in configs:
    model = MLPClassifier(**config['params'])
    start = time.time()
    model.fit(X_train, y_train)
    end = time.time()

    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='weighted')

    print(f"Model: {config['name']} | Time: {end-start:.2f}s | Val F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_model = model

print(f"\nBest model selected with Validation F1: {best_f1:.4f}")

Hyperparameter Optimization Results:
Model: Small Network | Time: 87.27s | Val F1: 0.8601
Model: Deep Network | Time: 148.61s | Val F1: 0.8507
Model: High Learning Rate | Time: 41.10s | Val F1: 0.8657

Best model selected with Validation F1: 0.8657


### 3.3. Evaluation on test set

In [ ]:
y_pred_test = best_model.predict(X_test)

print("Final Evaluation on Test Set:")
print(classification_report(y_test, y_pred_test, target_names=dataset['train'].features['label'].names))

Final Evaluation on Test Set:
              precision    recall  f1-score   support

     sadness       0.86      0.92      0.89       581
         joy       0.88      0.87      0.88       695
        love       0.72      0.70      0.71       159
       anger       0.85      0.82      0.84       275
        fear       0.85      0.76      0.80       224
    surprise       0.68      0.71      0.70        66

    accuracy                           0.85      2000
   macro avg       0.81      0.80      0.80      2000
weighted avg       0.85      0.85      0.85      2000



---

## 4. Results and summary

### 4.1 Corpus insights

The "Emotion" dataset originates from Twitter. To create this corpus without manual annotation, the authors used a "Distant Supervision" technique. They collected tweets containing emotion-related hashtags (#happy, #sad) and mapped them to 6 basic emotions. Crucially, they removed the hashtags from the text, forcing the model to learn the emotional context from the surrounding words rather than just memorizing the tags.

### 4.2 Results

My optimized MLP model, utilizing TF-IDF for text vectorization, achieved a weighted F1-score of around 0.85 on the test set. The model performed exceptionally well on predicting "joy" and "sadness", which are the majority classes, but struggled slightly with "surprise" and "love" due to the lower number of training samples for these specific emotions.

### 4.3 Relation to state of the art

The current state-of-the-art (SOTA) models for this dataset are Large Pre-trained Language Models (PLMs) such as RoBERTa, which achieve around 93%-95% accuracy. The gap of roughly 10-15% between my MLP model and the SOTA is expected. TF-IDF merely counts word frequencies and ignores word order/context, whereas Transformer models understand deep semantic relationships. However, the MLP approach remains highly computationally efficient and trains in seconds compared to the hours required for PLMs.

---

## 5. Test with out-of-domain documents

### 5.1. Annotating out-of-domain documents

**Description of out-of-domain documents:** I selected 50 sentences from global news headlines, financial reports, and historical events. This domain is completely different from the original corpus (Twitter) because news language is formal, objective, and complex, lacking the direct emotional slang found on social media.

**Process of annotation:** I manually read each of the 50 sentences and assigned one of the 6 emotion labels (0 to 5) that best matched the underlying sentiment of the event described ( assigning "0 - sadness" to disaster news, or "1 - joy" to economic success).

### 5.2 Conversion into dataset

In [ ]:
# 0: sadness, 1: joy, 2: love, 3: angry, 4: fear, 5: surprise

bonus_data = [
    ("Late last month, Food52 declared bankruptcy after years of financial struggles", 0),
    ("Hundreds of thousands of Nepalese were made homeless with entire villages flattened, across many districts of the country.", 0),
    ("Employment and social integration were strongly associated with deaths of despair.", 0),
    ("Weather conditions deteriorated sharply, so all rescue operations have been suspended.", 0),
    ("Former No. 1 overall pick and Stanley Cup champion defensive Erik Johnson announced his retirement from the NHL on Wednesday.", 0),
    ("When Rome burned in 192 CE, the city's vibrant community of scholars was devastated", 0),
    ("Frequent extremes damage infrastructure and ruin crops, destroying many of the investments farmers make at the start of each year.", 0),
    ("A Florida vice mayor seen as a rising political star was found dead at her home Wednesday.", 0),

    ("The company reported a pre-tax profit margin of 77%, marking the sixth consecutive quarter above 70%.", 1),
    ("BTS broke records with Arirang while bypassing traditional media.", 1),
    ("The Treaty of Versailles was a peace treaty signed on 28 June 1919.", 1),
    ("Psychologically, the triumph of a hometown team can bring immense joy and pride.", 1),
    ("Russia unveils a breakthrough cancer vaccine, aiming to transform treatment and survival outcomes.", 1),
    ("Sealine came alive last night with breathtaking fireworks lighting up the desert sky!", 1),
    ("The US economy has so far added factory jobs in just two of 10 months after Trump's return to office.", 1),
    ("BIGBANG has officially announced their 2026 World Tour, marking a special comeback as they celebrate their 20th anniversary.", 1),

    ("After 29 years of service, this woman heads off into retirement with this beautiful surprise ovation from students and faculty.", 2),
    ("Katie's main passion is advocating animal welfare issues while rescuing and rehabilitating animals and lovingly finding them forever homes.", 2),
    ("Dr Amy Beethe adopted a 5-year-old boy who arrived alone for heart surgery without parents or a caseworker.", 2),
    ("A tow truck driver in Washington state went the extra mile and hiked into a forest to help a driver who followed GPS and ended up stranded.", 2),
    ("Hospital staff got together to help a couple get married amid the bride's rare cancer diagnosis.", 2),
    ("A group of volunteers knitted thousands of blankets for NICU babies to keep them warm", 2),
    ("Alfred Street Baptist Church turned its Easter offering into life-changing relief for local families in public housing.", 2),
    ("Harold and Frances Pugh celebrated their 70th anniversary with the wedding they never had this past weekend", 2),

    ("Human rights groups are condemning a 'disturbing' rise in deaths within U.S. immigration detention centers.", 3),
    ("Miss Finland stripped of crown following apparent racist gesture.", 3),
    ("Sen. Tom Cotton faces backlash for repeatedly asking TikTok’s CEO about his citizenship.", 3),
    ("Activists were scheduled to rally outside Twitter headquarters Saturday morning to protest owner Elon Musk's layoffs of thousands of Twitter workers.", 3),
    ("The harsh criticism from the reviewers sparked a massive controversy online.", 3),
    ("Thousands rally in Imphal demanding justice for two children killed in a bombing, clashing with security forces amid protests.", 3),
    ("Pakistani capital in pandemic-style limbo as people work from home, public transport is closed and businesses shut.", 3),
    ("The U.S. military says it will blockade Iranian ports as Iran peace talks collapse.", 3),
    ("They've launched a hostile bid for the company with ever-sweetened offers that keep getting spurned.", 3),

    ("The local council decided it couldn't afford to protect the town from rising sea levels after 2054.", 4),
    ("Cybersecurity order warns of 'imminent risk' to federal agencies following possible breach.", 4),
    ("Analysis so far shows the outbreak is being caused by group B meningococcal bacteria.", 4),
    ("Social media shows fear, anxiety caused by Hurricane Michael.", 4),
    ("Private health records of half a million Britons offered for sale on Chinese website.", 4),
    ("One pupil dead in Vantaa school shooting.", 4),
    ("Covid-19 cases are rising again in much of the world.", 4),
    ("Malaysia Airlines flight 370 disappeared on March 8, 2014, en route from Kuala Lumpur to Beijing.", 4),
    ("Researchers are warning artificial intelligence has evolved.", 4),

    ("Scientists found a “lost world” of animals that shouldn’t exist yet.", 5),
    ("These shocking election results left political pundits dumbfounded.", 5),
    ("Scientists find huge pool of magna underneath central Italy.", 5),
    ("Five times an underdog has defied the odds.", 5),
    ("BTS was the first artists in a while from a non-Big-3 company to have won a Daesang, breaking the Big-3's monopoly.", 5),
    ("62-foot 'kraken-like' octopus identified as 'top-tier predator' 100M years ago.", 5),
    ("The world's fattest species of parrot - have had their most successful breeding season on record.", 5),
    ("DeepSeek unveils new AI model tailored for Huawei chips as China pushes for tech autonomy.", 5)
]


df_bonus = pd.DataFrame(bonus_data, columns=['text', 'label'])
X_bonus = vectorizer.transform(df_bonus['text'])
y_bonus = df_bonus['label']

### 5.3. Model evaluation on out-of-domain test set

In [ ]:
y_pred_bonus = best_model.predict(X_bonus)

print("Evaluation on Out-of-Domain Dataset (News):")
print(classification_report(y_bonus, y_pred_bonus, target_names=dataset['train'].features['label'].names))

Evaluation on Out-of-Domain Dataset (News):
              precision    recall  f1-score   support

     sadness       0.19      0.38      0.25         8
         joy       0.27      0.50      0.35         8
        love       0.00      0.00      0.00         8
       anger       0.22      0.22      0.22         9
        fear       0.22      0.22      0.22         9
    surprise       0.00      0.00      0.00         8

    accuracy                           0.22        50
   macro avg       0.15      0.22      0.17        50
weighted avg       0.15      0.22      0.18        50



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### 5.4 Results

As expected, the model's performance dropped significantly on the out-of-domain dataset compared to the original Twitter test set. This illustrates the concept of "Domain Shift". The model was trained on direct, first-person social media text ("I feel so sad"), so it struggles to interpret the complex, formal, and indirect expressions of emotions found in global news and historical events ("entire villages flattened").

### 5.5. Annotated data

In [ ]:
# 5.5 Display annotated data
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', None)
display(df_bonus)

,text,label
0,"Late last month, Food52 declared bankruptcy after years of financial struggles",0
1,"Hundreds of thousands of Nepalese were made homeless with entire villages flattened, across many districts of the country.",0
2,Employment and social integration were strongly associated with deaths of despair.,0
3,"Weather conditions deteriorated sharply, so all rescue operations have been suspended.",0
4,Former No. 1 overall pick and Stanley Cup champion defensive Erik Johnson announced his retirement from the NHL on Wednesday.,0
5,"When Rome burned in 192 CE, the city's vibrant community of scholars was devastated",0
6,"Frequent extremes damage infrastructure and ruin crops, destroying many of the investments farmers make at the start of each year.",0
7,A Florida vice mayor seen as a rising political star was found dead at her home Wednesday.,0
8,"The company reported a pre-tax profit margin of 77%, marking the sixth consecutive quarter above 70%.",1
9,BTS broke records with Arirang while bypassing traditional media.,1
